In [56]:
import numpy as np
import pandas as pd
import pandapower as pp
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pickle
import pp_heig_simulation as pp_sim
import pp_heig_plot as pp_plot
from datetime import time
import re

In [57]:
## Import pickle
net_trey = pp.from_pickle("input-data/trey_net_student.p")

In [58]:
### Line parameters
## Add a new column "length_km" to net_trey
net_trey.line.insert(
    7, "length_km", [0.180, 0.100, 0.115, 0.080, 0.090, 0.135, 0.100, 0.205, 0.085, 0.255, 0.340]
) 

## Create new column "cable_type" by removing the unwanted value (_x) use to differentiate each line from "name" with regex
# re.sub replaces the matches with the given substitute string
# First argument: regex pattern
# Second argument: substitute string
# Third argument: input string
net_trey.line["cable_type"] = net_trey.line.apply(
    lambda x: re.sub(r"_[0-9]$", "", x["name"]), axis=1
)
 
## Create parameter table for cable types
cable_types = pd.DataFrame(
    {
        "cable_type": ["GKN3x150_150", "GKN3X95_95", "GKN3X50_50", "GKT3X50_50","GKN3X240_240"],
        "r_ohm_per_km"  : [0.124, 0.193, 0.387, 0.387, 0.0754],
        "x_ohm_per_km"  : [0.07, 0.07, 0.07, 0.07, 0.07],
        "c_uf_per_km"   : [0.349, 0.338, 0.298, 0.298, 0.346],
        "max_i_ka"      : [0.400, 0.252, 0.170, 0.170, 0.512],
    }
)
 
## Merge net_trey.line with cable_types on "cable_type"
net_trey.line = net_trey.line.merge(cable_types, on="cable_type", how="left")

## Show updated net_trey.line
net_trey.line

,name,from_bus,to_bus,g_us_per_km,df,std_type,in_service,length_km,cable_type,r_ohm_per_km,x_ohm_per_km,c_uf_per_km,max_i_ka
0,GKN3x150_150,0,1,0.0,1.0,None,True,0.180,GKN3x150_150,0.1240,0.07,0.349,0.400
1,GKN3x150_150_2,1,2,0.0,1.0,None,True,0.100,GKN3x150_150,0.1240,0.07,0.349,0.400
2,GKN3X95_95,1,3,0.0,1.0,None,True,0.115,GKN3X95_95,0.1930,0.07,0.338,0.252
3,GKT3X50_50,3,4,0.0,1.0,None,True,0.080,GKT3X50_50,0.3870,0.07,0.298,0.170
4,GKT3X50_50_2,4,5,0.0,1.0,None,True,0.090,GKT3X50_50,0.3870,0.07,0.298,0.170
5,GKN3x150_150_3,0,6,0.0,1.0,None,True,0.135,GKN3x150_150,0.1240,0.07,0.349,0.400
6,GKN3X95_95_2,6,7,0.0,1.0,None,True,0.100,GKN3X95_95,0.1930,0.07,0.338,0.252
7,GKN3x150_150_4,7,8,0.0,1.0,None,True,0.205,GKN3x150_150,0.1240,0.07,0.349,0.400
8,GKN3X50_50,7,9,0.0,1.0,None,True,0.085,GKN3X50_50,0.3870,0.07,0.298,0.170
9,GKN3X50_50_2,6,10,0.0,1.0,None,True,0.255,GKN3X50_50,0.3870,0.07,0.298,0.170


In [59]:
## Buses parameters
net_trey.bus["vn_kv"] = net_trey.bus.apply(
    lambda row: 18.3 if row["type"] == "Slack" else (0.420 if row["type"] == "PQ" else None),
    axis=1,
)

net_trey.bus

,name,type,zone,in_service,vn_kv
0,STMT003438,PQ,Trafo,True,0.42
1,CDBT004764,PQ,North,True,0.42
2,CDBT003746,PQ,North,True,0.42
3,CDBT004760,PQ,North,True,0.42
4,CDBT012139,PQ,North,True,0.42
5,CDBT900784,PQ,North,True,0.42
6,CDBT901452,PQ,South,True,0.42
7,CDBT004774,PQ,South,True,0.42
8,CDBT901604,PQ,South,True,0.42
9,CDBT016055,PQ,South,True,0.42


In [60]:
## Connect external grid to bus 12
net_trey.ext_grid.insert(2, "bus", 12)

In [61]:
## Plot grid
pp_plot.plot_power_network(
    net=net_trey,
    plot_title="Trey",
    filename="trey_grid_example",
)

In [62]:
## Assign fixed loads from Excel file
profile_file_path = "input-data/power_profile.xlsx"

# Read the Excel file to get fixed load values
profile_df = pd.read_excel(profile_file_path, header=[0, 1])
profile_df = profile_df.dropna(how='all', axis=0)

# Take the first row of data as fixed load values
if len(profile_df) > 0:
    first_row = profile_df.iloc[0]
    
    # Extract P and Q for each load (assuming column structure: 0, P [MW], Q [MVAR], 1, P [MW], Q [MVAR], etc.)
    for load_idx in range(len(net_trey.load)):
        try:
            p_col = (load_idx, 'P [MW]')
            q_col = (load_idx, 'Q [MVAR]')
            if p_col in profile_df.columns and q_col in profile_df.columns:
                p_value = first_row[p_col]
                q_value = first_row[q_col]
                # Update load values
                net_trey.load.at[load_idx, 'p_mw'] = p_value
                net_trey.load.at[load_idx, 'q_mvar'] = q_value
        except Exception as e:
            print(f"Could not assign load {load_idx}: {e}")

print("Fixed loads assigned from Excel")
print(net_trey.load[['name', 'p_mw', 'q_mvar']])

Fixed loads assigned from Excel
          name  p_mw  q_mvar
0   STMT003438   NaN     NaN
1   CDBT004764   NaN     NaN
2   CDBT003746   NaN     NaN
3   CDBT004760   NaN     NaN
4   CDBT012139   NaN     NaN
5   CDBT900784   NaN     NaN
6   CDBT901452   NaN     NaN
7   CDBT004774   NaN     NaN
8   CDBT901604   NaN     NaN
9   CDBT016055   NaN     NaN
10          N1   NaN     NaN
11       60437   NaN     NaN


In [63]:
## Verify network configuration with a quick Load Flow
pp.runpp(net_trey)

print("Load Flow simulation completed successfully!")
print("\nBus voltages (V [pu]):")
print(net_trey.res_bus[['vm_pu', 'p_mw', 'q_mvar']])

print("\nLine loading (%):")
print(net_trey.res_line[['loading_percent']])

KeyError: 'parallel'

In [ ]:
## Inspect time_series structure
profile_file_path = "input-data/power_profile.xlsx"
time_series: dict = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

print("time_series keys:", time_series.keys())
for key in time_series:
    print(f"  {key}: {type(time_series[key])}")
    if isinstance(time_series[key], dict):
        print(f"    sub-keys: {list(time_series[key].keys())}")

print("\nnet_trey.load shape:", net_trey.load.shape)
print("net_trey.load columns:", net_trey.load.columns.tolist())

time_series keys: dict_keys([])

net_trey.load shape: (24, 11)
net_trey.load columns: ['name', 'profile_mapping', 'bus', 'const_z_percent', 'const_i_percent', 'sn_mva', 'in_service', 'type', 'p_mw', 'scaling', 'q_mvar']


In [ ]:
# Import profile data from Excel
profile_file_path = "input-data/power_profile.xlsx"
output_folder = ""
output_filename = "simulation_results"

# Load power profile from Excel file
time_series: dict = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

# Map each load row to the single available profile (index 0)
net_trey.load["profile_mapping"] = 0

pp_sim.apply_power_profile(net=net_trey, equipment="load", power_profiles=time_series["load"])

pp_sim.create_output_writer(net=net_trey)
result_df = pp_sim.run_time_simulation(
    net=net_trey,
    output_filename=output_filename,
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage",
    filename="voltage_result",
)

KeyError: 'load'